<a href="https://colab.research.google.com/github/laboratoriodecodigos/Colab-Python/blob/main/Conteo_de_pases_de_futbol_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⚽ Conteo de pases con IA · Laboratorio de Códigos
Todo en una sola celda. Activa **GPU T4**, ejecuta la celda, sube tu video y al final se descarga `pases_ia.mp4`.

`SALTO = 2` reproduce el video de prueba; usa `SALTO = 1` para analizar todos los cuadros.

In [ ]:
# =====================================================================
#  Conteo de pases con IA · Laboratorio de Códigos
#  YOLO (recortes a resolución nativa) + ByteTrack + K-Means + reglas de posesión
#  Uso en Colab: Entorno de ejecución > Cambiar tipo > GPU T4, pega este script en UNA celda y ejecútalo.
#  Te pedirá subir el video y al final descarga "pases_ia.mp4".
# =====================================================================
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics", "scikit-learn", "lap"], check=True)

import os, cv2, numpy as np
from collections import defaultdict, Counter
from tqdm.auto import tqdm
from ultralytics import YOLO
from sklearn.cluster import KMeans

try:
    from google.colab import files
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

# ---------------- parámetros generales ----------------
SALTO      = 2         # 2 = analiza 1 de cada 2 cuadros (así se generó el video de prueba). 1 = todos los cuadros
TILE       = 1280      # recorte a resolución nativa para buscar el balón
CONF_BALON = 0.15
TAM_BALON  = (10, 45)  # tamaño válido del balón en px (video 4K). En 1080p usa (5, 25)
SALIDA     = "pases_ia.mp4"

if EN_COLAB:
    subido = files.upload()
    VIDEO_IN = next(iter(subido))
else:
    VIDEO_IN = sys.argv[1]

modelo_balon = YOLO("yolo26m.pt")   # detecta el balón en recortes (predict)
modelo_jug   = YOLO("yolo26m.pt")   # sigue jugadores (track); instancia aparte para no reiniciar el tracker
PERSONA, BALON = 0, 32              # clases COCO



Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


Saving 14358056_3840_2160_24fps.mp4 to 14358056_3840_2160_24fps.mp4


In [ ]:

# =====================================================================
# 1. Herramientas: color de playera, filtro de cancha y búsqueda del balón
# =====================================================================

def color_torso(frame, caja):
    x1, y1, x2, y2 = map(int, caja)
    h, w = y2 - y1, x2 - x1
    rec = frame[y1 + int(0.2 * h): y1 + int(0.5 * h), x1 + int(0.25 * w): x2 - int(0.25 * w)]
    if rec.size == 0:
        return None
    hsv = cv2.cvtColor(rec, cv2.COLOR_BGR2HSV).reshape(-1, 3)
    pasto = (hsv[:, 0] > 35) & (hsv[:, 0] < 85) & (hsv[:, 1] > 50)
    pix = rec.reshape(-1, 3)[~pasto]
    if len(pix) < 15:
        return None
    lab = cv2.cvtColor(pix.reshape(-1, 1, 3).astype(np.uint8), cv2.COLOR_BGR2LAB).reshape(-1, 3)
    return np.median(lab, 0).astype(float)

def en_cancha(frame, x, y, r=40):
    H, W = frame.shape[:2]
    p = frame[max(0, int(y) - r):min(H, int(y) + r), max(0, int(x) - r):min(W, int(x) + r)]
    if p.size == 0:
        return False
    hsv = cv2.cvtColor(p, cv2.COLOR_BGR2HSV)
    pasto = (hsv[..., 0] > 30) & (hsv[..., 0] < 90) & (hsv[..., 1] > 40) & (hsv[..., 2] > 40)
    return pasto.mean() > 0.35

def x_tiles(w):
    xs = list(range(0, max(w - TILE, 0) + 1, TILE - 160))   # 160 px de traslape
    if xs[-1] + TILE < w:
        xs.append(w - TILE)
    return xs

def detectar_balon(frame, y0, ultimo):
    H, W = frame.shape[:2]
    if ultimo is not None:                       # recorte alrededor de la última posición
        cx, cy = ultimo
        x = int(np.clip(cx - TILE / 2, 0, max(W - TILE, 0)))
        y = int(np.clip(cy - TILE / 2, y0, max(H - TILE, y0)))
        regiones = [(x, y, frame[y:y + TILE, x:x + TILE])]
    else:                                         # barrido completo de la zona de la cancha
        regiones = [(x, y0, frame[y0:, x:x + TILE]) for x in x_tiles(W)]
    res = modelo_balon.predict([r[2] for r in regiones], classes=[BALON], imgsz=TILE,
                               conf=CONF_BALON, verbose=False)
    cands = []
    for (ox, oy, _), r in zip(regiones, res):
        for b, c in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.conf.tolist()):
            t = max(b[2] - b[0], b[3] - b[1])
            x, y = (b[0] + b[2]) / 2 + ox, (b[1] + b[3]) / 2 + oy
            if TAM_BALON[0] <= t <= TAM_BALON[1] and en_cancha(frame, x, y):
                cands.append((x, y, c, t))          # t = tamaño del balón en px (sirve para estimar profundidad)
    return cands

In [ ]:
# =====================================================================
# 2. Pasada 1 · detecciones (jugadores con ByteTrack + balón con recortes)
# =====================================================================

from tqdm.auto import tqdm

cap = cv2.VideoCapture(VIDEO_IN)
fps = cap.get(cv2.CAP_PROP_FPS) or 30
W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
Y0 = int(0.45 * H)                 # desde aquí hacia abajo está la cancha
IMGSZ_JUG = min(1920, W)           # jugadores: la franja completa a 1920 px basta

datos, ultimo, perdido = [], None, 99
for idx in tqdm(range(total)):
    ok, frame = cap.read()
    if not ok:
        break
    if idx % SALTO:
        continue
    # jugadores (solo personas en el tracker)
    r = modelo_jug.track(frame[Y0:], persist=True, tracker="bytetrack.yaml", classes=[PERSONA],
                         imgsz=IMGSZ_JUG, conf=0.25, verbose=False)[0]
    personas = []
    if r.boxes.id is not None:
        for b, tid, c in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.id.int().tolist(), r.boxes.conf.tolist()):
            b = b + np.array([0, Y0, 0, Y0])
            personas.append((tid, b, c, color_torso(frame, b)))
    # balón: primero cerca de donde iba; si no aparece, barrido completo
    cands = detectar_balon(frame, Y0, ultimo if perdido <= 3 else None)
    if not cands and perdido <= 3:
        cands = detectar_balon(frame, Y0, None)
    if cands:
        if ultimo is not None and perdido <= 6:   # preferir confianza alta Y cercanía a la posición previa
            best = max(cands, key=lambda c: c[2] - np.hypot(c[0] - ultimo[0], c[1] - ultimo[1]) / 800)
        else:
            best = max(cands, key=lambda c: c[2])
        ultimo, perdido = (best[0], best[1]), 0
    else:
        best = None
        perdido += 1
    datos.append(dict(cuadro=idx, personas=personas, balon=best))
cap.release()

fps_ef = fps / SALTO
print(f"Cuadros analizados: {len(datos)} | con balón: {sum(d['balon'] is not None for d in datos)}")



  0%|          | 0/464 [00:00<?, ?it/s]

Cuadros analizados: 232 | con balón: 177


In [ ]:
# =====================================================================
# 3. Equipos por color de playera (por cuadro, suavizado por ID)
# =====================================================================

VENTANA_EQUIPO_SEG = 0.5      # suavizado de la etiqueta de equipo (± este tiempo, mismo ID)

def rasgo(lab):
    L, A, B = lab
    return np.array([0.5 * L, 2.0 * (A - 128), 2.0 * (B - 128)])

# 1) K-Means con el color de todas las detecciones confiables
X = np.array([rasgo(col) for d in datos for tid, b, c, col in d["personas"] if col is not None and c > 0.4])
km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(X)
orden = list(np.argsort(-np.bincount(km.labels_, minlength=3)))
mapa = {orden[0]: 0, orden[1]: 1, orden[2]: -1}         # 0 = A, 1 = B, -1 = árbitro/otros

# 2) etiqueta cruda por detección
crudo = defaultdict(list)                                  # tid -> [(n, grupo o None)]
for n, d in enumerate(datos):
    for tid, b, c, col in d["personas"]:
        g = mapa[km.predict([rasgo(col)])[0]] if col is not None else None
        crudo[tid].append((n, g))

# 3) suavizado: mayoría de votos en la ventana del mismo ID
v = max(1, round(VENTANA_EQUIPO_SEG * fps_ef))
equipo_det = [dict() for _ in datos]                       # equipo_det[n][tid] = 0, 1 o -1
for tid, serie in crudo.items():
    for i, (n, _) in enumerate(serie):
        votos = Counter(g for m, g in serie[max(0, i - v): i + v + 1] if g is not None and abs(m - n) <= v)
        if not votos:   # sin color en la ventana: usar la mayoría del ID completo
            votos = Counter(g for _, g in serie if g is not None) or Counter({-1: 1})
        equipo_det[n][tid] = votos.most_common(1)[0][0]

colores_equipo = {}
for g in range(3):
    centro = km.cluster_centers_[g] / np.array([0.5, 2.0, 2.0]) + np.array([0, 128, 128])
    colores_equipo[mapa[g]] = tuple(int(x) for x in cv2.cvtColor(np.uint8([[np.clip(centro, 0, 255)]]), cv2.COLOR_LAB2BGR)[0, 0])

cambios = [t for t, s in crudo.items() if len({equipo_det[n][t] for n, _ in s}) > 1]
print("Colores BGR  A:", colores_equipo[0], " B:", colores_equipo[1], " árbitro/otros:", colores_equipo[-1])
print("IDs que el tracker pasó de un jugador a otro de distinto color:", cambios)

Colores BGR  A: (16, 15, 14)  B: (121, 77, 40)  árbitro/otros: (33, 87, 104)
IDs que el tracker pasó de un jugador a otro de distinto color: [7, 11, 37, 40, 70, 75, 82, 87, 92, 102]


In [ ]:
# =====================================================================
# 4. Posesión, pases y recuperaciones
# =====================================================================

MAX_HUECO_SEG = 0.6       # se interpola para dibujar y medir velocidad
HUECO_POSESION_SEG = 0.2  # solo huecos así de cortos sirven para decidir quién tiene el balón
POSESION_SEG  = 0.15
DIST_POSESION = 0.55     # zona de los pies: distancia horizontal máxima (en alturas del jugador)
DIST_VERT     = 0.3      # ... y vertical máxima (si se ve a la altura del torso, puede estar detrás)
DIST_CUERPO   = 0.35     # zona del cuerpo (pies, pecho, cabeza): distancia máxima a su caja
PROF_TOL      = (0.75, 1.6)  # el tamaño balón/jugador debe estar en este rango del valor típico
LIBRE_SEG     = 0.08     # en un pase el balón queda libre (lejos de todos) al menos este tiempo
DIST_MIN_PASE = 0.8
VEL_MAX       = 8.0
ID_NUEVO_SEG  = 0.5      # si el poseedor desaparece y en ese mismo momento nace un ID a su lado, es el mismo jugador
RECUP_SEG     = 0.4      # para un robo se exige un control más largo (evita forcejeos)
DIST_MARCA    = 1.0      # si el poseedor sigue a menos de 1 altura del balón, el rival solo lo está marcando
VEL_RECUP     = 3.0      # para robar, el rival debe frenar el balón (velocidad relativa < 3 alturas/s)
GIRO_TOQUE    = 25       # grados: cambio de dirección del balón que indica un toque
CAMBIO_TOQUE  = 0.4      # o cambio de rapidez del 40 %
CUADROS_POS   = max(1, round(POSESION_SEG * fps_ef))
CUADROS_REC   = max(1, round(RECUP_SEG * fps_ef))

# --- interpolar huecos cortos del balón ---
pos = [d["balon"][:2] if d["balon"] else None for d in datos]
max_hueco = round(MAX_HUECO_SEG * fps_ef)
i = 0
while i < len(pos):
    if pos[i] is None:
        j = i
        while j < len(pos) and pos[j] is None:
            j += 1
        if i > 0 and j < len(pos) and (j - i) <= max_hueco:
            a, b = np.array(pos[i - 1]), np.array(pos[j])
            for k in range(i, j):
                pos[k] = tuple(a + (b - a) * (k - i + 1) / (j - i + 1))
        i = j
    else:
        i += 1
# --- profundidad: tamaño del balón vs. altura del jugador ---
# Balón ≈ 22 cm, persona ≈ 1.75 m. Si están a la misma distancia de la cámara, su proporción en la imagen es constante.
# Un balón lejano que pasa "por detrás" de un jugador cercano se ve mucho más pequeño en proporción.
tam = [d["balon"][3] if d["balon"] else None for d in datos]
muestras = []
for n, d in enumerate(datos):
    if tam[n] is None:
        continue
    for tid, b, c, col in d["personas"]:
        h = b[3] - b[1]
        if abs(pos[n][0] - (b[0] + b[2]) / 2) < 0.4 * h and abs(pos[n][1] - b[3]) < 0.2 * h:   # balón en sus pies
            muestras.append(tam[n] / h)
PROP_BALON = float(np.median(muestras)) if len(muestras) >= 5 else 0.15
print(f"Proporción típica balón/jugador: {PROP_BALON:.3f} ({len(muestras)} muestras)")

def misma_profundidad(n, h):
    # True/False si se puede saber; None si no hay tamaño (cuadro interpolado)
    if tam[n] is None:
        return None
    r = tam[n] / h / PROP_BALON
    return PROF_TOL[0] <= r <= PROF_TOL[1]

# cuadros "fiables" para la posesión: detección real o hueco corto
fiable = [d["balon"] is not None for d in datos]
hp, i = round(HUECO_POSESION_SEG * fps_ef), 0
while i < len(fiable):
    if not fiable[i]:
        j = i
        while j < len(fiable) and datos[j]["balon"] is None:
            j += 1
        if (j - i) <= hp and pos[i] is not None:
            for k in range(i, j):
                fiable[k] = True
        i = j
    else:
        i += 1
print(f"Balón con posición tras interpolar: {sum(p is not None for p in pos)}/{len(pos)}")

# --- velocidad del balón relativa a un jugador ---
pies_id = [{tid: ((b[0] + b[2]) / 2, b[3]) for tid, b, c, col in d["personas"]} for d in datos]
def vel_rel(n, tid):
    a, b = max(n - 1, 0), min(n + 1, len(pos) - 1)
    if b == a or pos[a] is None or pos[b] is None or tid not in pies_id[a] or tid not in pies_id[b]:
        return 0.0
    vb = np.subtract(pos[b], pos[a]); vj = np.subtract(pies_id[b][tid], pies_id[a][tid])
    return np.hypot(*(vb - vj)) * fps_ef / (b - a)

# --- ¿alguien tocó el balón? Su trayectoria cambia de dirección o de rapidez ---
def vel_media(a, b):
    a, b = max(a, 0), min(b, len(pos) - 1)
    if b <= a or pos[a] is None or pos[b] is None:
        return None
    return np.subtract(pos[b], pos[a]) / (b - a)

def hubo_toque(i0, i1):
    pasos = max(2, round(0.2 * fps_ef))
    antes, despues = vel_media(i0 - pasos, i0), vel_media(i1 - pasos, i1)
    if antes is None or despues is None:
        return True                       # sin datos suficientes: no se puede descartar
    r0, r1 = np.linalg.norm(antes), np.linalg.norm(despues)
    if r0 < 1e-6 or r1 < 1e-6:
        return True                       # el balón se detuvo (o arrancó): hubo control
    giro = np.degrees(np.arccos(np.clip(np.dot(antes, despues) / (r0 * r1), -1, 1)))
    return giro > GIRO_TOQUE or abs(r1 / r0 - 1) > CAMBIO_TOQUE

# --- primer y último cuadro en que aparece cada ID ---
primer_visto, ultimo_visto = {}, {}
for n, d in enumerate(datos):
    for tid, *_ in d["personas"]:
        primer_visto.setdefault(tid, n)
        ultimo_visto[tid] = n
id_nuevo = round(ID_NUEVO_SEG * fps_ef)

# --- posesión, pases y recuperaciones ---
pases, recuperaciones = [0, 0], 0
duenio, candidato, racha, inicio = None, None, 0, 0
en_control = []            # quién tiene el balón en su zona en cada cuadro (None = balón libre)
min_libre = max(1, round(LIBRE_SEG * fps_ef))
eventos, estado = [], []
for n, d in enumerate(datos):
    jug = {}
    for tid, b, c, col in d["personas"]:
        jug[tid] = dict(id=tid, eq=equipo_det[n].get(tid, -1), caja=b, pies=((b[0] + b[2]) / 2, b[3]), alto=b[3] - b[1])
    if duenio and duenio["id"] in jug and jug[duenio["id"]]["eq"] == duenio["eq"]:   # mismo jugador, sigue visible
        duenio.update(pies=jug[duenio["id"]]["pies"], alto=jug[duenio["id"]]["alto"])

    if not fiable[n]:               # hueco largo interpolado: no se decide posesión con una posición estimada
        estado.append((dict(duenio) if duenio else None, list(pases), recuperaciones))
        en_control.append(None)
        continue
    cerca, bal = None, pos[n]
    if bal is not None:
        js = [j for j in jug.values() if j["eq"] >= 0]   # los árbitros no cuentan
        if js:
            def cercania(j):     # < 1 = el balón está en la zona de control del jugador
                h = j["alto"]
                prof = misma_profundidad(n, h)
                if prof is False:                       # balón más lejos (o más cerca) que el jugador
                    return 99.0
                if prof:                                # misma profundidad: vale todo el cuerpo (pies, pecho, cabeza)
                    x1, y1, x2, y2 = j["caja"]
                    dx, dy = max(x1 - bal[0], 0, bal[0] - x2), max(y1 - bal[1], 0, bal[1] - y2)
                    return np.hypot(dx, dy) / (DIST_CUERPO * h)
                dx, dy = bal[0] - j["pies"][0], bal[1] - j["pies"][1]      # sin tamaño: solo zona de los pies
                return np.hypot(dx / (DIST_POSESION * h), dy / (DIST_VERT * h))
            j = min(js, key=cercania)
            if cercania(j) < 1 and vel_rel(n, j["id"]) < VEL_MAX * j["alto"]:
                cerca = j
            if cerca and duenio and cerca["eq"] != duenio["eq"]:          # posible robo: condiciones extra
                d_own = jug.get(duenio["id"])
                marca = (d_own is not None and d_own["eq"] == duenio["eq"] and
                         np.hypot(d_own["pies"][0] - bal[0], d_own["pies"][1] - bal[1]) < DIST_MARCA * d_own["alto"])
                frena = vel_rel(n, cerca["id"]) < VEL_RECUP * cerca["alto"]
                if marca or not frena:
                    cerca = None

    en_control.append(cerca["id"] if cerca else None)
    if cerca and candidato and cerca["id"] == candidato["id"] and cerca["eq"] == candidato["eq"]:
        racha += 1
    else:
        racha, inicio = (1 if cerca else 0), n
    candidato = cerca

    necesita = CUADROS_REC if (duenio and candidato and candidato["eq"] != duenio["eq"]) else CUADROS_POS
    otro = duenio is None or candidato is None or candidato["id"] != duenio["id"] or candidato["eq"] != duenio["eq"]
    if otro and candidato and duenio and candidato["eq"] != duenio["eq"] and racha >= necesita:
        if not hubo_toque(inicio, n):     # el balón pasó rodando junto al rival sin que lo tocara
            otro = False
    if candidato and racha >= necesita and otro:
        if duenio is not None:
            sep = np.hypot(candidato["pies"][0] - duenio["pies"][0], candidato["pies"][1] - duenio["pies"][1])
            alto = (candidato["alto"] + duenio["alto"]) / 2
            # cambio de ID: el poseedor desapareció justo cuando nació el ID del "receptor"
            ultimo_dueno = max([m for m in range(n) if duenio["id"] in pies_id[m]] or [-999])
            cambio_id = (duenio["id"] not in jug and n - primer_visto[candidato["id"]] <= id_nuevo
                         and abs(primer_visto[candidato["id"]] - ultimo_dueno) <= id_nuevo)
            ult = max([m for m in range(len(en_control)) if en_control[m] == duenio["id"]] or [-1])
            libre = sum(1 for m in range(ult + 1, inicio) if en_control[m] is None) >= min_libre
            if candidato["eq"] == duenio["eq"]:
                if sep > DIST_MIN_PASE * alto and libre and not cambio_id:
                    pases[duenio["eq"]] += 1
                    salida = datos[max(ult, 0)]["cuadro"]
                    eventos.append(("PASE", duenio["eq"], duenio["pies"], candidato["pies"], d["cuadro"], salida))
            else:
                recuperaciones += 1
                eventos.append(("RECUPERA", candidato["eq"], duenio["pies"], candidato["pies"], d["cuadro"], d["cuadro"]))
        duenio = dict(candidato)
    estado.append((dict(duenio) if duenio else None, list(pases), recuperaciones))

for tipo, eq, _, _, llega, sale in eventos:
    print(f"{tipo:9s} equipo {'AB'[eq]}   sale {sale / fps:5.2f}s → llega {llega / fps:5.2f}s")
print(f"\nPases equipo A: {pases[0]} | Pases equipo B: {pases[1]} | Recuperaciones: {recuperaciones}")

Proporción típica balón/jugador: 0.158 (52 muestras)
Balón con posición tras interpolar: 211/232
PASE      equipo B   sale  3.33s → llega  4.33s
PASE      equipo B   sale  5.50s → llega  7.50s
PASE      equipo B   sale  9.42s → llega 11.67s
PASE      equipo B   sale 11.83s → llega 15.25s
PASE      equipo B   sale 15.25s → llega 16.92s
PASE      equipo B   sale 17.00s → llega 19.00s

Pases equipo A: 0 | Pases equipo B: 6 | Recuperaciones: 0


In [ ]:

# =====================================================================
# 5. Dibujar el resultado
# =====================================================================

ESC = min(1.0, 1920 / W)
Wo, Ho = int(W * ESC), int(H * ESC)
cap = cv2.VideoCapture(VIDEO_IN)
out = cv2.VideoWriter("pases_raw.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (Wo, Ho))
P = lambda p: (int(p[0] * ESC), int(p[1] * ESC))
k = 0
for idx in tqdm(range(total)):
    ok, frame = cap.read()
    if not ok:
        break
    while k + 1 < len(datos) and datos[k + 1]["cuadro"] <= idx:
        k += 1
    frame = cv2.resize(frame, (Wo, Ho))
    d, (duenio, cuenta, rec) = datos[k], estado[k]
    for tid, b, c, col in d["personas"]:
        eq = equipo_det[k].get(tid, -1)
        pies, ancho = P(((b[0] + b[2]) / 2, b[3])), int((b[2] - b[0]) * ESC / 2)
        cv2.ellipse(frame, pies, (ancho, max(4, ancho // 3)), 0, -45, 235, colores_equipo[eq], 3)
        if duenio and tid == duenio["id"]:
            cv2.putText(frame, "CON BALON", P((b[0], b[1] - 12)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    for tipo, eq, p1, p2, c0, _ in eventos:
        if 0 <= idx - c0 < fps * 1.5:
            color = colores_equipo[eq] if tipo == "PASE" else (0, 0, 255)
            cv2.arrowedLine(frame, P(p1), P(p2), (255, 255, 255), 7, tipLength=0.08)   # contorno blanco
            cv2.arrowedLine(frame, P(p1), P(p2), color, 3, tipLength=0.08)
            cv2.putText(frame, tipo, P(p2), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 5)
            cv2.putText(frame, tipo, P(p2), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
    if pos[k] is not None:
        x, y = P(pos[k])
        cv2.drawMarker(frame, (x, y - 18), (0, 255, 255), cv2.MARKER_TRIANGLE_DOWN, 18, 3)
    cv2.rectangle(frame, (0, 0), (Wo, 60), (30, 30, 30), -1)
    for x0, eq, txt in [(20, 0, f"Pases A: {cuenta[0]}"), (Wo // 3, 1, f"Pases B: {cuenta[1]}")]:
        cv2.circle(frame, (x0 + 15, 30), 15, colores_equipo[eq], -1)
        cv2.circle(frame, (x0 + 15, 30), 15, (255, 255, 255), 2)
        cv2.putText(frame, txt, (x0 + 40, 42), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 3)
    cv2.putText(frame, f"Recuperaciones: {rec}", (2 * Wo // 3, 42), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 3)
    out.write(frame)
cap.release(); out.release()

subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", "pases_raw.mp4", "-c:v", "libx264",
                "-preset", "veryfast", "-pix_fmt", "yuv420p", SALIDA], check=True)
os.remove("pases_raw.mp4")
print(f"\nVideo listo: {SALIDA}")

if EN_COLAB:
    from base64 import b64encode
    from IPython.display import HTML, display
    datos_video = b64encode(open(SALIDA, "rb").read()).decode()
    display(HTML(f'<video width=720 controls autoplay loop><source src="data:video/mp4;base64,{datos_video}" type="video/mp4"></video>'))
    files.download(SALIDA)